In [4]:
# Configuración de entorno
ENTORNO = "colab" # Opciones: "local" o "colab"
ALMACENAMIENTO = "drive" # Opciones: "local" o "drive" (solo aplica si ENTORNO = "colab")

# Rutas base según la configuración
RUTA_DRIVE = '/content/drive/MyDrive/TFM'
RUTA_LOCAL = 'C:/Users/Usuario/Desktop/TFM_Mateu/gnn-material-science' # o ruta absoluta si se desea

import os
import sys

if ENTORNO == "colab":
    print("Ejecutando en entorno Google Colab...")
    dispositivo = "cuda"
    if ALMACENAMIENTO == "drive":
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            os.chdir(RUTA_DRIVE)
            print(f"Montado Drive y trabajando en {RUTA_DRIVE}")
        except ImportError:
            print("No se pudo montar drive. ¿Seguro que estás en Colab?")
    else:
        print("Usando almacenamiento local de la sesión de Colab.")
    
    # ✅ Instalar dependencias ANTES de cambiar el directorio de trabajo
    print("Instalando dependencias...")
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         os.path.join(RUTA_DRIVE, "requirements.txt")],
        cwd="/content"  # Forzamos que pip trabaje desde /content, no desde Drive
    )
    
    # ✅ Cambiar al directorio del proyecto DESPUÉS de instalar
    os.chdir(RUTA_DRIVE)
    print(f"Directorio de trabajo: {os.getcwd()}")

    """# Instalamos dependencias si estamos en Colab
    print("Comprobando e instalando dependencias (puede tardar un poco)...")
    get_ipython().system('pip install -q -r requirements.txt')
    """
    get_ipython().system('pip install -q cuequivariance cuequivariance-torch cuequivariance-ops-torch-cu12')
elif ENTORNO == "local":
    os.chdir(RUTA_LOCAL)
    dispositivo = "cpu" # Forzamos CPU para local según especificaciones
    print(f"Ejecutando en entorno local usando {dispositivo.upper()}. Directorio actual: {os.getcwd()}")
    if 'google.colab' in sys.modules:
        print("ADVERTENCIA: Parece que estás en Colab pero has seleccionado entorno 'local'.")
    
    print("Comprobando e instalando dependencias locales (puede tardar un poco)...")
    get_ipython().system('pip install -q -r requirements.txt')
else:
    print("Entorno no reconocido. Usa 'colab' o 'local'.")
    dispositivo = "cpu"


Ejecutando en entorno Google Colab...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Montado Drive y trabajando en /content/drive/MyDrive/TFM
Instalando dependencias...
Directorio de trabajo: /content/drive/MyDrive/TFM


In [5]:
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA'}")

CUDA disponible: True
GPU: Tesla T4


In [6]:
# ============================================================
# PARAMETROS DE SIMULACION
# ============================================================
pasos = 5000

# Modo de progreso:
#   "log" -> muestra la tabla de energia/temperatura cada step (bueno para diagnostico)
#   "bar" -> muestra una barra de progreso visual (mas limpio para simulaciones largas)
modo_progreso = "bar"  # Opciones: "log" o "bar"

import sys
import subprocess

comando = [
    sys.executable, "-u", "MACE/npt.py",
    "--device", dispositivo,
    "--steps", str(pasos),
    "--progress", modo_progreso,
]
print(f"Ejecutando: {" ".join(comando)}\n")

# Usamos subprocess para forzar la impresion linea a linea en tiempo real
proceso = subprocess.Popen(comando, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for linea in proceso.stdout:
    print(linea, end="")

proceso.wait()
if proceso.returncode != 0:
    print(f"\nError: El proceso termino con codigo {proceso.returncode}")


Ejecutando: /usr/bin/python3 -u MACE/npt.py --device cuda --steps 5000 --progress bar

/usr/local/lib/python3.12/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1480: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Found 1 materials to process: ['BaLiF3']

Processing material: BaLiF3
Found structure at: /content/drive/MyDrive/TFM/input/candidates/BaLiF3/Pm-3m/POSCAR
Structure is too small (5 atoms). Target is 150.
Creating a 4x4x4 supercell...
Final structure size for MD: 320 atoms
Saved supercell reference to /content/drive/MyDrive/TFM/MACE/results/BaLiF3/POSCAR-supercell-4x4x4
Using f